# RoadSafe India: Exploratory Data Analysis & Road Safety KPIs
## Data-Driven Road Safety Analytics for Safer & More Sustainable Urban Planning

**Project:** IBM & AICTE Internship Project  
**Author:** Aarya (Computer Engineering)  
**Data Source:** Ministry of Road Transport & Highways (MoRTH) Transport Research Wing (TRW)  

---
### Objectives of this Notebook:
1. Ingest raw MoRTH accident datasets (State multi-year trends, 50 Million-Plus Cities, Contributing Factors, Diurnal Time Patterns).
2. Execute reproducible data cleaning and standardization.
3. Compute standardized road safety indicators (**Accident Severity Index**, **Injury Ratio**, **VRU Share**).
4. Visualize temporal trajectories and urban vulnerability clusters.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_cleaner import DataCleaner
from src.kpi_calculator import KPICalculator

# Set visual style
sns.set_theme(style="whitegrid")
print("Environment initialized successfully!")

## 1. Execute Data Cleaning & Standardization Pipeline

In [ ]:
cleaner = DataCleaner(raw_dir="../data/raw", processed_dir="../data/processed")
df_state = cleaner.clean_state_ut_data()
df_cities = cleaner.clean_million_plus_cities()
df_factors = cleaner.clean_contributing_factors()
df_time = cleaner.clean_time_slots()

print(f"State/UT Records: {df_state.shape}")
print(f"50 Million-Plus Cities Records: {df_cities.shape}")
print(f"Contributing Factors: {df_factors.shape}")
print(f"Time Slots: {df_time.shape}")

## 2. National Multi-Year Trend Analysis (2014–2020)

In [ ]:
nat_trend = df_state.groupby("Year")[["Total_Accidents", "Persons_Killed", "Persons_Injured"]].sum().reset_index()
nat_trend["Severity_Index"] = (nat_trend["Persons_Killed"] / nat_trend["Total_Accidents"]) * 100
nat_trend

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5), dpi=150)
ax1.plot(nat_trend["Year"], nat_trend["Total_Accidents"] / 1000, marker="o", color="#1f77b4", linewidth=2.5, label="Total Accidents ('000)")
ax1.set_xlabel("Year", fontsize=11, fontweight="bold")
ax1.set_ylabel("Total Accidents in Thousands", color="#1f77b4", fontsize=11, fontweight="bold")

ax2 = ax1.twinx()
ax2.plot(nat_trend["Year"], nat_trend["Persons_Killed"] / 1000, marker="s", color="#d62728", linewidth=2.5, linestyle="--", label="Fatalities ('000)")
ax2.set_ylabel("Fatalities in Thousands", color="#d62728", fontsize=11, fontweight="bold")

plt.title("India Road Safety Trends (2014-2020): Accidents vs Fatalities", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. 50 Million-Plus Cities: Crash Volume vs Severity Index

In [ ]:
plt.figure(figsize=(12, 6), dpi=150)
sns.scatterplot(
    data=df_cities,
    x="Total_Accidents",
    y="Severity_Index",
    size="Persons_Killed",
    hue="Severity_Index",
    palette="Reds",
    sizes=(40, 400),
    legend=False
)

# Annotate top cities
for idx, row in df_cities.head(10).iterrows():
    plt.text(row["Total_Accidents"] + 40, row["Severity_Index"], row["City"], fontsize=9, fontweight="semibold")

plt.title("50 Million-Plus Cities: Crash Volume vs Accident Severity Index (2020)", fontsize=13, fontweight="bold")
plt.xlabel("Total Recorded Accidents", fontsize=11, fontweight="bold")
plt.ylabel("Severity Index (Fatalities per 100 Crashes)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Vulnerable Road Users (VRUs) Vulnerability Breakdown

In [ ]:
vru_data = df_factors[df_factors["Category_Type"] == "Road User Type"].sort_values(by="Persons_Killed", ascending=False)

plt.figure(figsize=(8, 8), dpi=150)
plt.pie(
    vru_data["Persons_Killed"],
    labels=vru_data["Factor_Name"],
    autopct="%1.1f%%",
    startangle=140,
    colors=sns.color_palette("tab10")
)
plt.title("Fatality Distribution Across Road User Categories (2020)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()